### Predictive Modelling

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au
####  *DATE last modified:* 11/05/2026

This notebook applies machine learning to identify mineralised regions within arc–backarc environments. Using a Random Forest model trained on features primarily extracted along trench lines in subduction zones, we investigate the factors that drive mineralisation and the locations where deposits are most likely to form. The analysis focuses on four key aspects: training a predictive model to locate mineralisation, determining which features contribute most to deposit formation, quantifying uncertainty to highlight areas of higher and lower confidence, and enabling users to assess model accuracy. Finally, the predictions are mapped through geological time in an interactive map, providing a practical tool to support exploration efforts.

We begin by importing the required libraries:

In [ ]:
# Import libraries
from ipywidgets import interact
import os

import cartopy.crs as ccrs
from gplately import PlateModelManager
import joblib
from joblib import Parallel, delayed
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import shap
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, partial_dependence, permutation_importance
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real
from tqdm import tqdm

# Note: Ensure the 'lib' folder is located in the same folder as this notebook
from lib.predictive_modelling import *
from lib.feature_extraction import *
from lib.plot import *

# Load configuration parameters (e.g., paths, model names)
from parameters import parameters

### Setup

As defined in `parameters.py`, the cell below configures the analysis parameters and specifies the paths to input/output files and directories. You can also specify the number of cores to use by setting an appropriate value for the `n_jobs` variable at the end of the cell.

**Note:** You can modify the analysis settings directly in `parameters.py`, located in the same folder as this notebook. The file is structured as a dictionary; look for keys such as `timespan` and `temporal_resolution` to adjust their values as needed.

In [ ]:
# Set up temporal analysis parameters
plate_model = parameters["plate_model"]
anchor_plate_id = parameters["anchor_plate_id"]

temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"] # Youngest time to analyse
time_max = parameters["timespan"]["max"] # Oldest time to analyse
# Create an array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

buffer_distance = parameters["buffer_distance"]
columns_to_drop_backarc = parameters["columns_to_drop_backarc"]

# Directory paths for inputs and outputs
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
figs_dir = parameters["figs_dir"]
ml_dir = parameters["ml_dir"]
backarc_prob_grids_dir = parameters["backarc_prob_grids_dir"]
backarc_uncertainty_grids_dir = parameters["backarc_uncertainty_grids_dir"]
backarc_prob_unc_grids_dir = parameters["backarc_prob_unc_grids_dir"]
backarc_prob_maps_dir = parameters["backarc_prob_maps_dir"]
backarc_uncertainty_maps_dir = parameters["backarc_uncertainty_maps_dir"]
backarc_prob_unc_maps_dir = parameters["backarc_prob_unc_maps_dir"]

# Data filenames
Xy_train_new_filename = parameters["Xy_train_new_filename"]
Xy_rf_train_filename = parameters["Xy_rf_train_filename"]
Xy_rf_val_filename = parameters["Xy_rf_val_filename"]
Xy_rf_test_filename = parameters["Xy_rf_test_filename"]
Xy_pos_test_filename = parameters["Xy_pos_test_filename"]
gini_importances_filename = parameters["gini_importances_filename"]
permutation_importances_filename = parameters["permutation_importances_filename"]
shap_values_filename = parameters["shap_values_filename"]
features_shap_test_filename = parameters["features_shap_test_filename"]
backarc_data_filename = parameters["backarc_data_filename"]
backarc_prob_filename = parameters["backarc_prob_filename"]
deposit_coords_recon_filename = parameters["deposit_coords_recon_filename"]
deposit_coords_recon_all_filtered_filename = parameters["deposit_coords_recon_all_filtered_filename"]

model_rf_filename = parameters["model_rf_filename"]
model_rf_calibrated_filename = parameters["model_rf_calibrated_filename"]
robust_scaler_filename = parameters["robust_scaler_filename"]

# Construct full directory paths
figs_dir = os.path.join(outputs_dir, figs_dir)
ml_dir = os.path.join(outputs_dir, ml_dir)
backarc_prob_grids_dir = os.path.join(ml_dir, backarc_prob_grids_dir)
backarc_uncertainty_grids_dir = os.path.join(ml_dir, backarc_uncertainty_grids_dir)
backarc_prob_unc_grids_dir = os.path.join(ml_dir, backarc_prob_unc_grids_dir)
backarc_prob_maps_dir = os.path.join(ml_dir, backarc_prob_maps_dir)
backarc_uncertainty_maps_dir = os.path.join(ml_dir, backarc_uncertainty_maps_dir)
backarc_prob_unc_maps_dir = os.path.join(ml_dir, backarc_prob_unc_maps_dir)

# Construct full file paths
Xy_train_new_filename = os.path.join(ml_dir, Xy_train_new_filename)
Xy_rf_train_filename = os.path.join(ml_dir, Xy_rf_train_filename)
Xy_rf_val_filename = os.path.join(ml_dir, Xy_rf_val_filename)
Xy_rf_test_filename = os.path.join(ml_dir, Xy_rf_test_filename)
Xy_pos_test_filename = os.path.join(ml_dir, Xy_pos_test_filename)
gini_importances_filename = os.path.join(ml_dir, gini_importances_filename)
permutation_importances_filename = os.path.join(ml_dir, permutation_importances_filename)
shap_values_filename = os.path.join(ml_dir, shap_values_filename)
features_shap_test_filename = os.path.join(ml_dir, features_shap_test_filename)
backarc_data_filename = os.path.join(outputs_dir, backarc_data_filename)
backarc_prob_filename = os.path.join(ml_dir, backarc_prob_filename)
deposit_coords_recon_filename = os.path.join(outputs_dir, deposit_coords_recon_filename)
deposit_coords_recon_all_filtered_filename = os.path.join(outputs_dir, deposit_coords_recon_all_filtered_filename)

model_rf_filename = os.path.join(ml_dir, model_rf_filename)
model_rf_calibrated_filename = os.path.join(ml_dir, model_rf_calibrated_filename)
robust_scaler_filename = os.path.join(ml_dir, robust_scaler_filename)

agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")

# Number of cores to be used for running this notebook
n_jobs = 20

### Random Forest

We train a machine learning model to distinguish between mineralised and barren areas. The Random Forest algorithm is chosen because it effectively handles complex geological relationships and diverse types of data.

The key steps include:
- Removing positive samples identified through the positive-unlabelled bagging method and retaining only those random samples labelled as negative.  
- Balancing the dataset by ensuring an equal number of positive and negative samples, preventing model bias.  
- Splitting the data into training, validation, and testing sets for proper evaluation.  
- Tuning hyperparameters to optimise model performance.  
- Calibrating the model so that its predictions better capture real-world uncertainty.

In [ ]:
# Load and prepare training data
Xy_train_new = pd.read_csv(Xy_train_new_filename)
# Xy_train_new["label"] = Xy_train_new["label"].replace(2, 1)
# Remove samples labelled as 2 (positive samples identified by the 
# positive-unlabelled bagging method) to improve model reliability 
# by training only on definitive samples.
Xy_train_new = Xy_train_new[Xy_train_new["label"] != 2]

# Address imbalance by matching positive and negative sample sizes
# This prevents the model from being biased toward the majority class
Xy_train_new_ones = Xy_train_new[Xy_train_new["label"] == 1]
Xy_train_new_zeros = Xy_train_new[Xy_train_new["label"] == 0]
Xy_train_new_zeros_sampled = Xy_train_new_zeros.sample(n=len(Xy_train_new_ones), random_state=42)
Xy_train_new = pd.concat([Xy_train_new_ones, Xy_train_new_zeros_sampled])
Xy_train_new = Xy_train_new.sample(frac=1, random_state=42).reset_index(drop=True)

print("Number of positive samples:", len(Xy_train_new_ones))
print("Number of negative samples:", len(Xy_train_new_zeros_sampled))

# Separate features from labels
features_rf = Xy_train_new.copy()
features_rf = features_rf.drop("label", axis=1)
labels_rf = Xy_train_new["label"]

if os.path.isfile(model_rf_calibrated_filename):
    # Load existing model and data if available
    model_rf = joblib.load(model_rf_filename)
    model_rf_calibrated = joblib.load(model_rf_calibrated_filename)
    Xy_rf_train = pd.read_csv(Xy_rf_train_filename)
    Xy_rf_val = pd.read_csv(Xy_rf_val_filename)
    Xy_rf_test = pd.read_csv(Xy_rf_test_filename)
else:
    # Random Forest model structure
    rf = RandomForestClassifier(n_jobs=n_jobs, random_state=42)

    # Split into train (60%), validation (20%), and test (20%)
    X_rf_train_temp, X_rf_test, y_rf_train_temp, y_rf_test = train_test_split(features_rf, labels_rf, test_size=0.2, random_state=42)
    # Further split training data to create a validation set
    X_rf_train, X_rf_val, y_rf_train, y_rf_val = train_test_split(X_rf_train_temp, y_rf_train_temp, test_size=0.25, random_state=42)
    
    # Save the training, validation, and test sets to CSV files
    Xy_rf_train = np.hstack((X_rf_train, y_rf_train.values.reshape(-1, 1)))
    Xy_rf_train = pd.DataFrame(Xy_rf_train, columns=Xy_train_new.columns)
    Xy_rf_train.to_csv(Xy_rf_train_filename, index=False)
    
    Xy_rf_val = np.hstack((X_rf_val, y_rf_val.values.reshape(-1, 1)))
    Xy_rf_val = pd.DataFrame(Xy_rf_val, columns=Xy_train_new.columns)
    Xy_rf_val.to_csv(Xy_rf_val_filename, index=False)
    
    Xy_rf_test = np.hstack((X_rf_test, y_rf_test.values.reshape(-1, 1)))
    Xy_rf_test = pd.DataFrame(Xy_rf_test, columns=Xy_train_new.columns)
    Xy_rf_test.to_csv(Xy_rf_test_filename, index=False)

    # Extract sample weights
    weights_rf_train = Xy_rf_train["weight"]
    X_rf_train = Xy_rf_train[[col for col in Xy_rf_train.columns if col not in ["weight", "label"]]]
    
    weights_rf_val = Xy_rf_val["weight"]
    X_rf_val = Xy_rf_val[[col for col in Xy_rf_val.columns if col not in ["weight", "label"]]]
    
    weights_rf_test = Xy_rf_test["weight"]
    X_rf_test = Xy_rf_test[[col for col in Xy_rf_test.columns if col not in ["weight", "label"]]]

    # Bayesian search for hyperparameters
    search_space = {
    "bootstrap": Categorical([True, False]),
    "max_depth": Integer(5, 30),
    "max_features": Categorical([None, "sqrt","log2"]), 
    "min_samples_leaf": Integer(2, 10),
    "min_samples_split": Integer(2, 20),
    "n_estimators": Integer(50, 300)
    }

    rf_bayes_search = BayesSearchCV(
        rf,
        search_space,
        n_iter=50, # specify how many iterations
        scoring="f1", # Use F1 score for evaluation
        n_jobs=n_jobs,
        cv=5, # Number of cross-validation folds
        verbose=1,
        random_state=42
    )
    
    # Fit the model using training data
    rf_bayes_search.fit(X_rf_train, y_rf_train, sample_weight=weights_rf_train)
    
    # Extract the optimisation results
    optimization_results = rf_bayes_search.cv_results_["mean_test_score"]
    
    model_rf = rf_bayes_search.best_estimator_
    model_rf_f1 = rf_bayes_search.best_score_    
    print("The highest F1-score during cross validation:", model_rf_f1)
    
    # Evaluate uncalibrated model on validation set
    uncalibrated_probs = model_rf.predict_proba(X_rf_val)[:, 1]
    uncalibrated_brier = brier_score_loss(y_rf_val, uncalibrated_probs, sample_weight=weights_rf_val)
    print("Uncalibrated Brier Score (validation set):", uncalibrated_brier)

    # Calibrate using the validation data
    model_rf_calibrated = CalibratedClassifierCV(model_rf, method="isotonic", cv="prefit")
    model_rf_calibrated.fit(X_rf_val, y_rf_val, sample_weight=weights_rf_val)
    
    # Save the original and calibrated model
    joblib.dump(model_rf, model_rf_filename)
    joblib.dump(model_rf_calibrated, model_rf_calibrated_filename)
    
    # Evaluate both models on the test set
    uncal_test_probs = model_rf.predict_proba(X_rf_test)[:, 1]
    cal_test_probs = model_rf_calibrated.predict_proba(X_rf_test)[:, 1]
    
    uncal_brier = brier_score_loss(y_rf_test, uncal_test_probs, sample_weight=weights_rf_test)
    cal_brier = brier_score_loss(y_rf_test, cal_test_probs, sample_weight=weights_rf_test)
    
    print("Test Set Results:")
    print("Uncalibrated Brier Score:", uncal_brier)
    print("Calibrated Brier Score:", cal_brier)
    print(f"Improvement: {(uncal_brier - cal_brier) / uncal_brier * 100:.2f}%")

    # Calculate calibration curves
    uncal_fraction, uncal_predicted = calibration_curve(y_rf_test, uncal_test_probs, n_bins=20)
    cal_fraction, cal_predicted = calibration_curve(y_rf_test, cal_test_probs, n_bins=20)
    
    # Create side-by-side subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 1 row, 3 columns
    
    # Plot 1: Bayesian Optimisation Progress
    axes[0].plot(range(1, len(optimization_results) + 1),
                optimization_results, marker="o", color="black", markerfacecolor="red")
    axes[0].set_xlim(0, len(optimization_results) + 1)
    axes[0].set_xlabel("Bayesian Optimisation Iteration")
    axes[0].set_ylabel("Mean Test F1 Score")
    axes[0].set_title("Bayesian Optimisation Progress")
    axes[0].grid(True, linestyle=":")
    
    # Plot 2: Reliability Curve (Uncalibrated)
    axes[1].plot(uncal_predicted, uncal_fraction, marker="o", 
                label=f"Uncalibrated (Brier: {uncal_brier:.4f})")
    axes[1].plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    axes[1].set_xlabel("Mean predicted probability")
    axes[1].set_ylabel("Fraction of positives")
    axes[1].legend()
    axes[1].grid(True)
    axes[1].set_title("Uncalibrated Reliability Curve")
    
    # Plot 3: Reliability Curve (Calibrated)
    axes[2].plot(cal_predicted, cal_fraction, marker="o", 
                label=f"Calibrated (Brier: {cal_brier:.4f})")
    axes[2].plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    axes[2].set_xlabel("Mean predicted probability")
    axes[2].set_ylabel("Fraction of positives")
    axes[2].legend()
    axes[2].grid(True)
    axes[2].set_title("Calibrated Reliability Curve")
    
    plt.tight_layout()
    fig_path = "./Outputs/Figs/rf_bayesian_optimisation_progress.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

After training, we evaluate the model on unseen test data to assess its performance and determine how reliable its predictions are for real-world exploration.

In [ ]:
# Prepare test data for evaluation
features_rf_test = Xy_rf_test.copy()
features_rf_test = features_rf_test.drop(["weight", "label"], axis=1)
labels_rf_test = Xy_rf_test["label"]
weights_rf_test = Xy_rf_test["weight"]

# Predict using the calibrated model
labels_pred = model_rf_calibrated.predict(features_rf_test)

# Calculate and print comprehensive performance metrics
rf_cmatrix = confusion_matrix(labels_rf_test, labels_pred)
rf_acc = accuracy_score(labels_rf_test, labels_pred, sample_weight=weights_rf_test)
rf_pre = precision_score(labels_rf_test, labels_pred, sample_weight=weights_rf_test)
rf_rec = recall_score(labels_rf_test, labels_pred, sample_weight=weights_rf_test)
rf_f1 = f1_score(labels_rf_test, labels_pred, sample_weight=weights_rf_test)

print("Confusion matrix:\n", rf_cmatrix)
print("Accuracy:", rf_acc)
print("Precision:", rf_pre)
print("Recall:", rf_rec)
print("F1-Score:", rf_f1)

In [ ]:
# Additional validation on positive-only test set
# This evaluation is more reliable as it focuses on known deposits
Xy_pos_test = pd.read_csv(Xy_pos_test_filename)
X_pos_test = Xy_pos_test[[col for col in Xy_pos_test.columns if col not in ["weight", "label"]]]
y_pos_test = Xy_pos_test["label"]
weights_pos_test = Xy_pos_test["weight"]
X_pos_pred = model_rf_calibrated.predict(X_pos_test)
X_pos_pred_acc = accuracy_score(y_pos_test, X_pos_pred, sample_weight=weights_pos_test)
print("Accuracy:", X_pos_pred_acc)

### ROC Plot

We evaluate the model's ability to distinguish between mineralised and non-mineralised areas by testing its predictions across different thresholds. The ROC (Receiver Operating Characteristic) curve illustrates the trade-off between:

- **Sensitivity** – the proportion of actual deposits correctly identified.  
- **Specificity** – the proportion of non-mineralised samples correctly ignored.

This analysis helps identify the optimal threshold where predictions of mineralisation are both meaningful and reliable for exploration.

In [ ]:
# Generate probability predictions for ROC analysis
labels_prob = model_rf_calibrated.predict_proba(features_rf_test)
labels_name = ["Non-mineralised", "Mineralised"]
# Create ROC plot
fig_path = os.path.join(figs_dir, "rf_roc.png")
roc_plot(
    y_test = labels_rf_test,
    z_test = labels_prob, 
    n_classes = 2,
    labels_name = labels_name,
    average = "macro",
    fig_path = fig_path
)

### Feature Importance (Gini Importance)

Feature importance is a metric used by tree-based models like Random Forest to estimate the relative contribution of each feature to the model's predictions. It is computed by averaging the total reduction in Gini impurity (a measure of node impurity) brought by each feature across all trees in the forest. Features that more frequently and more significantly reduce impurity are considered more important. This helps in understanding which variables have the most influence in the model's decision-making process. We split the samples into five groups and train ten models using the same hyperparameters obtained through Bayesian optimisation. For each model, four groups are used for training while the remaining group is reserved for testing. The Gini importance of each feature is then calculated across all models, resulting in fifty values per feature. These values are visualised in a box plot, and the features are ranked from top to bottom according to their median importance.

In [ ]:
if os.path.isfile(gini_importances_filename):
    gini_importances = pd.read_csv(gini_importances_filename)
else:    
    # Setup
    n_splits = 5
    n_repeats = 10
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,
    )
    feature_names = list(features_rf_test.columns)
    feature_names = [clean_feature_name(i) for i in feature_names]
    gini_importances_all = []

    # Cross-validate with the best estimator
    for idx, _ in cv.split(features_rf, labels_rf):
        features_fold = features_rf.iloc[idx]
        weights_fold = features_fold["weight"]
        features_fold = features_fold.drop("weight", axis=1)
        labels_fold = labels_rf.iloc[idx]

        # Clone and fit
        clf = clone(model_rf_calibrated.estimator)
        clf.fit(features_fold, labels_fold, sample_weight=weights_fold)

        # Extract per-tree importances
        tree_importances = pd.DataFrame(
            [tree.feature_importances_ for tree in clf.estimators_],
            columns=feature_names
        )
        gini_importances_all.append(tree_importances)

    # Combine all results into one DataFrame
    gini_importances = pd.concat(gini_importances_all, ignore_index=True)

    # Sort columns by median importance
    gini_importances = gini_importances[gini_importances.median().sort_values(ascending=False).index]

    # Save to CSV
    
    gini_importances.to_csv(gini_importances_filename, index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

# Reverse column order so the most important is on top
gini_importances.iloc[:, ::-1].boxplot(vert=False, ax=ax)

# Customise plot
ax.grid(linestyle="dotted")
ax.set_xlim(0)
ax.set_xlabel("Gini Importance", fontsize=16)
ax.set_ylabel("Feature", rotation=0, ha="right", fontsize=16)
ax.yaxis.set_label_coords(-0.017, 1.02)

# Format y-tick labels
ax.set_yticks(
    ax.get_yticks(),
    [format_feature_name(label.get_text()) for label in ax.get_yticklabels()]
)

# Style tick labels
ax.tick_params(labelsize=12)

# plt.tight_layout()

fig_path = os.path.join(figs_dir, "gini_importance.png")
if not os.path.exists(fig_path):
    fig.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

### Permutation Importance

Permutation importance is a model-agnostic technique used to evaluate the impact of each feature on a model's predictive performance. It works by randomly shuffling the values of a single feature in the test data and measuring how much the model's performance decreases. If permuting a feature's values significantly degrades the model's score (e.g., F1 score), it indicates that the feature is important. This method provides a more direct assessment of feature relevance than internal metrics like Gini importance, especially in the presence of feature interactions or correlations. Similar to the previous step, we split the samples into five groups and train ten models using the hyperparameters obtained through Bayesian optimisation. In each model, four groups are used for training and the remaining group is held out for testing. The permutation importance of each feature is then computed across all models, yielding fifty values per feature. These values are displayed in a box plot, and the features are ranked from top to bottom based on their median importance.

In [ ]:
if os.path.isfile(permutation_importances_filename):
    permutation_importances = pd.read_csv(permutation_importances_filename)
else:
    # Setup
    n_splits = 5
    n_repeats = 10
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,
    )
    feature_names = list(features_rf_test.columns)
    feature_names = [clean_feature_name(i) for i in feature_names]
    permutation_importances_all = []

    # Cross-validate with the best estimator
    for train_idx, test_idx in cv.split(features_rf, labels_rf):
        features_train = features_rf.iloc[train_idx]
        weights_train = features_train["weight"]
        features_train = features_train.drop("weight", axis=1)
        labels_train = labels_rf.iloc[train_idx]
        features_test = features_rf.iloc[test_idx]
        weights_test = features_test["weight"]
        features_test = features_test.drop("weight", axis=1)
        labels_test = labels_rf.iloc[test_idx]

        # Clone and fit
        clf = clone(model_rf_calibrated.estimator)
        clf.fit(features_train, labels_train, sample_weight=weights_train)

        result = permutation_importance(clf, features_test, labels_test, sample_weight=weights_test, scoring="f1", n_repeats=10, random_state=42, n_jobs=n_jobs)
        result_df = pd.DataFrame(result.importances.T, columns=feature_names)
        permutation_importances_all.append(result_df)

    # Combine all results into one DataFrame
    permutation_importances = pd.concat(permutation_importances_all, ignore_index=True)

    # Sort columns by median importance
    permutation_importances = permutation_importances[permutation_importances.median().sort_values(ascending=False).index]

    # Save to CSV
    permutation_importances.to_csv(permutation_importances_filename, index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

# Reverse column order so the most important is on top
permutation_importances.iloc[:, ::-1].boxplot(vert=False, ax=ax)

# Customise plot
ax.grid(linestyle="dotted")
ax.set_xlabel("Permutation Importance", fontsize=16)
ax.set_ylabel("Feature", rotation=0, ha="right", fontsize=16)
ax.yaxis.set_label_coords(-0.017, 1.02)

# Format y-tick labels
ax.set_yticks(
    ax.get_yticks(),
    [format_feature_name(label.get_text()) for label in ax.get_yticklabels()]
)

# Style tick labels
ax.tick_params(labelsize=12)

# plt.tight_layout()

fig_path = os.path.join(figs_dir, "permutation_importance.png")
if not os.path.exists(fig_path):
    fig.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

### Partial Dependence and Individual Conditional Expectation

Partial Dependence Plots (PDPs) illustrate how individual features influence the model's predictions while averaging over all other features, making them useful for identifying general trends related to mineralisation. In contrast, Individual Conditional Expectation (ICE) curves show how predictions vary for individual samples, helping to detect unusual behaviour in specific zones. Both plots demonstrate how changes in a single feature affect the predicted probability of mineralisation while keeping other features constant. This cell is interactive, allowing you to select any input feature and explore its impact on model predictions—both on average and at the level of individual data points.

In [ ]:
# Prepare data for PDP/ICE analysis
features_rf_train = Xy_rf_train.copy()
labels_rf_train = features_rf_train["label"]
weights_rf_train = features_rf_train["weight"]
features_rf_train = features_rf_train.drop(["weight", "label"], axis=1)
feature_names = features_rf_train.columns
# Load scaler to convert back to original units
robust_scaler = joblib.load(robust_scaler_filename)
features_rf_train_original_values = features_rf_train * robust_scaler.scale_ + robust_scaler.center_
@interact
def show_pdp_ice(feature=feature_names):

    # --- fontsize controls (tweak as needed) ---
    label_fontsize = 14
    tick_fontsize = 12
    legend_fontsize = 12
    title_fontsize = 16

    plt.figure(figsize=(8, 6))
    # Compute partial dependence and ICE
    pd_result = partial_dependence(
        estimator=model_rf_calibrated.estimator,
        X=features_rf_train,
        features=[feature],
        sample_weight=weights_rf_train,
        kind="both", # 'both' shows PDP and ICE
        method="brute",
        grid_resolution=100,
        percentiles=(0.01, 0.99),
    )
    averaged = pd_result.average[0]
    individual = pd_result.individual[0]
    grid_values = pd_result.grid_values[0]
    # Plot ICE with label-based coloring
    for i, ice_line in enumerate(individual):
        label = labels_rf_train.iloc[i]
        color = "DarkSeaGreen" if label == 1 else "LightSalmon"
        plt.plot(grid_values, ice_line, color=color, alpha=0.3, linewidth=0.7)
    # Plot PDP
    plt.plot(grid_values, averaged, color="black", linewidth=2, linestyle="--")
    # Main x-axis ticks (outward, labeled, default spacing)
    plt.tick_params(axis="x", direction="out", length=6)
    # Add secondary inward ticks for percentiles (unlabelled)
    percentiles = np.linspace(1, 99, 11)
    percentile_values = np.percentile(features_rf_train_original_values[feature], percentiles)
    feature_min = features_rf_train_original_values[feature].quantile(0.01)
    feature_max = features_rf_train_original_values[feature].quantile(0.99)
    x_min = grid_values.min()
    x_max = grid_values.max()
    ax = plt.gca()
    plt.xlim(x_min, x_max)
    inward_tick_positions = (percentile_values - feature_min) / (feature_max - feature_min) * (x_max - x_min) + x_min
    for tick in inward_tick_positions:
        plt.axvline(x=tick, ymin=0, ymax=0.03, color="black", linewidth=1)
    locator = MaxNLocator(nbins=8, prune="both")
    ticks_original = locator.tick_values(feature_min, feature_max)
    
    ticks_scaled = (
        (ticks_original - feature_min)
        / (feature_max - feature_min)
        * (x_max - x_min)
        + x_min
    )
    ax.set_xticks(ticks_scaled)
    ax.set_xticklabels([f"{t:.0f}" for t in ticks_original])
        
    # Axis labelling and title
    plt.xlabel(format_feature_name(feature), fontsize=label_fontsize)
    plt.ylabel("Partial dependence", fontsize=label_fontsize)
    # plt.title(f"Partial Dependence and ICE for '{format_feature_name(feature)}'", fontsize=title_fontsize)
    ax.tick_params(labelsize=tick_fontsize)
    
    # Add legend
    pos_line = Line2D([], [], color="DarkSeaGreen", alpha=0.6, label="Positive")
    neg_line = Line2D([], [], color="LightSalmon", alpha=0.6, label="Negative")
    avg_line = Line2D([], [], color="black", linestyle="--", linewidth=2, label="Average")
    plt.legend(handles=[pos_line, neg_line, avg_line], loc="upper left", fontsize=legend_fontsize)
    plt.tight_layout()
    clean_feature = clean_feature_name(feature)
    fig_path = os.path.join(figs_dir, f"pd_{clean_feature}.png")
    # if not os.path.exists(fig_path):
    #     plt.savefig(
    #         fig_path,
    #         dpi=300,
    #         bbox_inches="tight",
    #     )
    
    plt.show()

### SHAP Values

SHAP (SHapley Additive exPlanations) values provide a unified measure of feature importance by quantifying how much each feature contributes to a specific prediction. Based on cooperative game theory, SHAP values fairly distribute the difference between a model's prediction and the average prediction among all input features. Unlike methods that only give global importance, SHAP values offer both global and local interpretability, showing not just which features are important, but also how they influence individual predictions—positively or negatively. We split the samples into five groups and train two models using the hyperparameters obtained through Bayesian optimisation. In each model, four groups are used for training and the remaining group is held out for testing. SHAP values for each feature is then computed across all models and the features are ranked from top to bottom based on their importance.

In [ ]:
if os.path.isfile(shap_values_filename):
    # Load existing SHAP values and features if available
    shap_values = pd.read_csv(shap_values_filename)
    shap_values_array = shap_values.values
    features_shap_test = pd.read_csv(features_shap_test_filename)
    feature_names = list(features_rf_test.columns)
    feature_names = [clean_feature_name(i) for i in feature_names]
else:
    # Setup
    n_splits = 5
    n_repeats = 2
    total_folds = n_splits * n_repeats
    
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,
    )
    
    test_idx_all = []
    shap_values_all = []

    # Cross-validate with the best estimator
    for train_idx, test_idx in tqdm(cv.split(features_rf, labels_rf), total=total_folds, desc="Computing SHAP"):
        features_train = features_rf.iloc[train_idx]
        weights_train = features_train["weight"]
        features_train = features_train.drop("weight", axis=1)
        labels_train = labels_rf.iloc[train_idx]
        
        features_test = features_rf.iloc[test_idx]
        weights_test = features_test["weight"]
        features_test = features_test.drop("weight", axis=1)
        labels_test = labels_rf.iloc[test_idx]
        test_idx_all.extend(test_idx)

        # Clone and fit
        clf = clone(model_rf_calibrated.estimator)
        clf.fit(features_train, labels_train, sample_weight=weights_train)

        # Compute SHAP values on the test set
        explainer = shap.TreeExplainer(clf, features_train)  # Use training data as background
        result = explainer(features_test, check_additivity=False)
        shap_values_all.append(result)
    
    # Combine all SHAP values from all folds
    shap_values_array = np.vstack([sv.values[:, :, 1] for sv in shap_values_all])
    feature_names = list(features_rf_test.columns)
    feature_names = [clean_feature_name(i) for i in feature_names]
    shap_values = pd.DataFrame(shap_values_array, columns=feature_names)
    
    # Save SHAP values and features for future use
    shap_values.to_csv(shap_values_filename, index=False)
    features_shap_test = features_rf.iloc[test_idx_all].drop("weight", axis=1)
    features_shap_test.to_csv(features_shap_test_filename, index=False)

feature_names = [format_feature_name(name) for name in feature_names]

# Create a SHAP explanation object for visualisation
explanation = shap.Explanation(
    values=shap_values_array,
    data=features_shap_test.values,
    feature_names=feature_names
)

# Plot SHAP values using a beeswarm plot
shap.plots.beeswarm(explanation, max_display=20, show=False)
fig = plt.gcf()

fig_path = os.path.join(figs_dir, "shap.png")
if not os.path.exists(fig_path):
    fig.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

### Calculate Probability, Uncertainty, and Adjusted Probability

We apply the trained model to the previously generated points within buffer zones that encompass arc–back arc environments to estimate:

- **Mineralisation Probability**: The model's confidence that geological conditions are favourable for deposit formation.  
- **Entropy**: A measure of prediction uncertainty derived from the probability distribution (high entropy indicates greater model uncertainty).  
- **Vote Variance**: The variability among decision trees in the random forest ensemble (high variance suggests conflicting predictions between trees).  
- **Uncertainty**: A combined uncertainty metric calculated from the normalised entropy and vote variance values. This metric captures both probabilistic uncertainty and disagreement within the ensemble model.  
- **Adjusted Probability**: A probability score weighted by uncertainty, calculated by reducing the predicted mineralisation probability in areas where uncertainty is high. This provides a more conservative estimate of mineralisation potential.

The resulting predictions are saved as global grids, highlighting where mineralisation is most likely and where the model is more or less confident. These grids make it possible to visualise spatial variations in mineralisation probability, helping to identify both prospective exploration targets and regions where predictions should be interpreted with greater caution.

In [ ]:
if os.path.isfile(backarc_prob_filename):
    print(f"Probabilities have already been calculated and saved in {backarc_prob_filename}\nLoading ...")
    backarc_prob = pd.read_csv(backarc_prob_filename)
else:
    output_cols = [
        "lon",
        "lat",
        "present_lon",
        "present_lat",
        "age (Ma)",
    ]

    # Get the same feature names used in model training
    selected_features = list(model_rf_calibrated.feature_names_in_)

    # Load points within arc–back arc environments along with their associated
    # feature values for prediction
    needed_cols = list(dict.fromkeys(output_cols + selected_features))
    backarc_data = pd.read_csv(backarc_data_filename, usecols=needed_cols)
    backarc_data_out = backarc_data[output_cols].copy()
    backarc_data_scaled = backarc_data.reindex(columns=selected_features)

    # Scale feature values using the same scaler as training
    backarc_data_scaled = robust_scaler.transform(backarc_data_scaled)
    backarc_data_scaled = pd.DataFrame(backarc_data_scaled, columns=selected_features)

    # Calculate probability and entropy values for all points
    # Entropy quantifies uncertainty in model predictions
    probs = model_rf_calibrated.predict_proba(backarc_data_scaled)[:, 1].ravel()
    backarc_prob = backarc_data_out.copy()
    backarc_prob["probability"] = probs
    entropies = calculate_entropy(probs)
    backarc_prob["entropy"] = entropies
    
    # Calculate vote variance
    # Vote variance shows disagreement among decision trees
    chunk_size = 100000  # Adjust based on available memory
    n_samples = backarc_data_scaled.shape[0]
    n_chunks = (n_samples // chunk_size) + (1 if n_samples % chunk_size != 0 else 0)  # Ensure last chunk is included
    
    for i in tqdm(range(n_chunks)):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, n_samples)
    
        X_chunk = backarc_data_scaled.iloc[start_idx:end_idx]
        
        # Calculate vote variance for the chunk
        vote_variances_chunk = calculate_tree_vote_variance(model_rf_calibrated.estimator, X_chunk.values)
        
        # Ensure that the chunk size matches the number of rows being processed
        if len(vote_variances_chunk) != (end_idx - start_idx):
            raise ValueError(f"Mismatch in size: {len(vote_variances_chunk)} vs. {end_idx - start_idx}")
        
        # Assign the result to the appropriate slice of the dataframe
        backarc_prob.loc[start_idx:end_idx-1, "vote_variance"] = vote_variances_chunk

    # Normalise entropy and vote variance to [0, 1]
    entropy_min = backarc_prob["entropy"].min()
    entropy_max = backarc_prob["entropy"].max()
    variance_min = backarc_prob["vote_variance"].min()
    variance_max = backarc_prob["vote_variance"].max()

    backarc_prob["entropy_norm"] = (
        (backarc_prob["entropy"] - entropy_min) / (entropy_max - entropy_min)
        if entropy_max > entropy_min else 0.0
    )

    backarc_prob["vote_variance_norm"] = (
        (backarc_prob["vote_variance"] - variance_min) / (variance_max - variance_min)
        if variance_max > variance_min else 0.0
    )

    # Combined uncertainty
    backarc_prob["uncertainty"] = (
        0.5 * backarc_prob["entropy_norm"]
        + 0.5 * backarc_prob["vote_variance_norm"]
    )

    # Probability adjusted by uncertainty
    backarc_prob["probability_uncertainty"] = (
        backarc_prob["probability"] * (1 - backarc_prob["uncertainty"])
    )
    
    # Export rows with calculated metrics
    print(f"Saving the metrics as a CSV file: {backarc_prob_filename}")
    backarc_prob.to_csv(backarc_prob_filename, index=False)

# Create probability grids
if os.path.exists(backarc_prob_grids_dir):
    print(f"Probability grids are located in {backarc_prob_grids_dir}")
else:
    os.makedirs(backarc_prob_grids_dir, exist_ok=True)
    
    create_grids(
        data=backarc_prob_filename,
        output_dir=backarc_prob_grids_dir,
        times=time_steps,
        extent=(-180, 180, -90, 90),
        threads=n_jobs,
        verbose=True,
        column="probability",
        filename_format="backarc_probability_grid_{}Ma.nc",        
    )
    
# Create uncertainty grids
if os.path.exists(backarc_uncertainty_grids_dir):
    print(f"Uncertainty grids are located in {backarc_uncertainty_grids_dir}")
else:    
    os.makedirs(backarc_uncertainty_grids_dir, exist_ok=True)
    
    create_grids(
        data=backarc_prob_filename,
        output_dir=backarc_uncertainty_grids_dir,
        times=time_steps,
        extent=(-180, 180, -90, 90),
        threads=n_jobs,
        verbose=True,
        column="uncertainty",
        filename_format="backarc_uncertainty_grid_{}Ma.nc",        
    )
    
# Create adjusted probability grids
if os.path.exists(backarc_prob_unc_grids_dir):
    print(f"Adjusted probability grids are located in {backarc_prob_unc_grids_dir}")
else:    
    os.makedirs(backarc_prob_unc_grids_dir, exist_ok=True)
    
    create_grids(
        data=backarc_prob_filename,
        output_dir=backarc_prob_unc_grids_dir,
        times=time_steps,
        extent=(-180, 180, -90, 90),
        threads=n_jobs,
        verbose=True,
        column="probability_uncertainty",
        filename_format="backarc_prob_unc_grid_{}Ma.nc",        
    )

### Probability Maps

The cell below loads the required files to create the `PlateReconstruction` and `PlateTopologies` objects, which will later be used to map mineralisation probability, uncertainty, and adjusted probability through geological time at a global scale.

In [ ]:
# Plate motion model
pmm = PlateModelManager()
pm = pmm.get_model(plate_model)

rotation_model = pm.get_rotation_model()
topology_features = pm.get_topologies()

static_polygons = pm.get_static_polygons()
coastlines = pm.get_coastlines()
continents = pm.get_continental_polygons()
COBs = pm.get_COBs()

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
    anchor_plate_id=anchor_plate_id,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
    anchor_plate_id=anchor_plate_id,
)

# Define map projection (Mollweide provides a good global view) and central meridian
projection = ccrs.Mollweide(central_longitude=30)
central_meridian = 0

This section creates an interactive probability map showing mineralisation probability along trench lines, together with known mineral occurrences. It also plots seafloor age and plate boundaries, including mid-ocean ridges and transform faults, at a selected time. The visualisation highlights spatial patterns, showing how mineralisation probability relates to tectonic features, and temporal evolution, showing how favourable conditions migrate through time.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            backarc_prob_grids_dir,
            f"backarc_probability_grid_{time:0.0f}Ma.nc", # Mineralisation probability grid
        ),
        grid_type="probability",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a full sequence of mineralisation probability maps through geological time and compiles them into a time-lapse animation, illustrating how the modelled probability of mineralisation changes over time.

In [ ]:
# Generate all probability maps and create animation
if os.path.exists(backarc_prob_maps_dir):
    print(f"Probability maps are located in {backarc_prob_maps_dir}")
else:
    os.makedirs(backarc_prob_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=n_jobs,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(backarc_prob_maps_dir, f"backarc_probability_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                anchor_plate_id,
                grid=os.path.join(
                    backarc_prob_grids_dir,
                    f"backarc_probability_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="probability",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the evolution of mineralisation probability
    # Animation reveals temporal patterns in geological processes
    output_filename = os.path.join(ml_dir, "backarc_probability_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Uncertainty Maps

This section generates interactive uncertainty maps along trench lines, combined with known mineral occurrences. The maps also display reconstructed seafloor age and plate boundaries—including mid-ocean ridges and transform faults—at a selected geological time step.

Uncertainty is calculated as the average of the normalised entropy and normalised vote variance:

$$
\text{Uncertainty} =
\frac{
\text{Entropy}_{\text{norm}} +
\text{Vote Variance}_{\text{norm}}
}{2}
$$

Entropy is a statistical measure of uncertainty in the model's probability predictions:

- **Low entropy (≈ 0):** The model is confident. Predictions lean strongly toward either a high or low probability of mineralisation.
- **High entropy (≈ 1):** The model is uncertain. Probabilities cluster near 50%, indicating ambiguity about whether geological conditions favour or hinder deposit formation.

Vote variance represents model disagreement within the Random Forest ensemble. Each decision tree casts a prediction for a given location, and the variance of these predictions is calculated across all trees.

- **Low vote variance:** Strong agreement between trees, indicating stable and consistent predictive signals.
- **High vote variance:** Greater disagreement between trees, suggesting uncertainty in how geological features relate to mineralisation potential.

By combining entropy and vote variance, the uncertainty metric captures both probabilistic ambiguity and ensemble disagreement, providing a more comprehensive measure of model confidence.

Uncertainty maps allow exploration of how spatial uncertainty in mineralisation predictions evolves through geological time. By stepping through reconstructed time intervals, it becomes possible to identify regions where predictions remain consistently robust, as well as areas where uncertainty is elevated, and interpretations should therefore be treated with greater caution.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            backarc_uncertainty_grids_dir,
            f"backarc_uncertainty_grid_{time:0.0f}Ma.nc", # Uncertainty grid
        ),
        grid_type="uncertainty",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a complete sequence of uncertainty maps through geological time and compiles them into a time-lapse animation. This visualisation shows how uncertainty evolves over time, providing an intuitive way to communicate model uncertainty across Earth's history. Such animations are especially useful for identifying when and where the predictive model is most stable or most variable.

In [ ]:
# Generate all uncertainty maps and create animation
if os.path.exists(backarc_uncertainty_maps_dir):
    print(f"Uncertainty maps are located in {backarc_uncertainty_maps_dir}")
else:
    os.makedirs(backarc_uncertainty_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=n_jobs,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(backarc_uncertainty_maps_dir, f"backarc_uncertainty_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                anchor_plate_id,
                grid=os.path.join(
                    backarc_uncertainty_grids_dir,
                    f"backarc_uncertainty_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="uncertainty",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the changes in uncertainty through geological time
    output_filename = os.path.join(ml_dir, "backarc_uncertainty_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Adjusted Probability Maps

This section generates interactive adjusted probability maps, where the predicted mineralisation probability is weighted by model uncertainty.

Adjusted probability is calculated as:

$$
\text{Adjusted Probability} =
\text{Probability} \times (1 - \text{Uncertainty})
$$

This reduces probability values in areas where uncertainty is high, producing a more conservative estimate of mineralisation potential. These maps highlight regions where the model predicts favourable geological conditions with greater confidence, while down-weighting areas where predictions are less certain.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            backarc_prob_unc_grids_dir,
            f"backarc_prob_unc_grid_{time:0.0f}Ma.nc", # Adjusted probability grid
        ),
        grid_type="probability_unc",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a full sequence of adjusted probability maps through geological time and compiles them into a time-lapse animation. These maps combine predicted mineralisation probability with model uncertainty, highlighting regions where favourable geological conditions are predicted with greater confidence while down-weighting areas of higher uncertainty at each time step.

In [ ]:
# Generate all adjusted probability maps and create animation
if os.path.exists(backarc_prob_unc_maps_dir):
    print(f"Adjusted probability maps are located in {backarc_prob_unc_maps_dir}")
else:
    os.makedirs(backarc_prob_unc_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=n_jobs,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(backarc_prob_unc_maps_dir, f"backarc_prob_unc_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                anchor_plate_id,
                grid=os.path.join(
                    backarc_prob_unc_grids_dir,
                    f"backarc_prob_unc_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="probability_unc",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the changes in adjusted probability through geological time
    output_filename = os.path.join(ml_dir, "backarc_probability_unc_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Trend of Average Probability, Entropy, and Vote Variance

This section visualises how the average predicted probability of porphyry mineralisation for points within the arc-backarc environments varies over geological time, and relates these probabilities to both the number of known deposits at different ages (shown as a histogram) and the average value of a selected feature associated with those probabilities. This highlights how the feature evolves through time.

It also provides the option to plot the uncertainty envelope for the average trends by calculating the standard deviation of values at each time step. The curves are smoothed to emphasise trends, and dual y-axes enable direct comparison between probabilities, deposit frequency, and feature values, providing a comprehensive temporal overview. In doing so, the visualisation reveals how the predicted probability of mineralisation changes across Earth's history, helping to identify key time periods favourable for porphyry formation.

In [ ]:
# Load data
selected_features = model_rf_calibrated.feature_names_in_
deposit_coords_recon = pd.read_csv(deposit_coords_recon_filename)

# Probability stats
backarc_prob_mean_age = (
    backarc_prob.groupby("age (Ma)")["probability"]
    .mean()
    .reset_index()
)

backarc_prob_unc_mean_age = (
    backarc_prob.groupby("age (Ma)")["probability_uncertainty"]
    .mean()
    .reset_index()
)

backarc_prob_mean_age = backarc_prob_mean_age.sort_values("age (Ma)")
backarc_prob_unc_mean_age = backarc_prob_unc_mean_age.sort_values("age (Ma)")

ages = backarc_prob_mean_age["age (Ma)"].values.reshape(-1, 1)
probs = backarc_prob_mean_age["probability"].values.reshape(-1, 1)
probs_unc = backarc_prob_unc_mean_age["probability_uncertainty"].values.reshape(-1, 1)

# Smooth mean once (does not depend on spread type)
ages_smooth, probs_smooth = smooth_curve(
    ages, probs, method="gaussian", fwhm_myr=25
)

_, probs_unc_smooth = smooth_curve(
    ages, probs_unc, method="gaussian", fwhm_myr=25
)

# Backarc feature data
selected_cols = np.append(selected_features, "age (Ma)")
backarc_data = pd.read_csv(backarc_data_filename, usecols=selected_cols)

def filter_data(df):
    return df[
        # (df["convergence_obliquity (degrees)"].between(-90, 90)) &
        (df["convergence_rate (cm/yr)"] >= 0) &
        (df["convergence_rate_orthogonal (cm/yr)"] >= 0)
        # (df["slab_flux (m^2/yr)"] >= 0) &
        # (df["subduction_water_flux_lithosphere (t/m/yr)"] >= 0)
    ]

distance_threshold = np.deg2rad(buffer_distance) * EARTH_RADIUS
backarc_data_filtered = backarc_data[
    backarc_data["distance_to_trench (km)"] <= distance_threshold
]
backarc_data_filtered = filter_data(backarc_data_filtered)

bin_width = 10
bins = np.arange(time_min, time_max + bin_width, bin_width)

# Interactive plot
@interact
def show_map(feature=selected_features, spread_method=["std", "mad"]):
    # Probability spread
    backarc_prob_spread_age = (
        backarc_prob.groupby("age (Ma)")["probability"]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name="probability")
        .sort_values("age (Ma)")
    )

    backarc_prob_unc_spread_age = (
        backarc_prob.groupby("age (Ma)")["probability_uncertainty"]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name="probability_uncertainty")
        .sort_values("age (Ma)")
    )

    probs_spread = backarc_prob_spread_age["probability"].values.reshape(-1, 1)
    probs_unc_spread = backarc_prob_unc_spread_age["probability_uncertainty"].values.reshape(-1, 1)

    _, probs_spread_smooth = smooth_curve(
        ages, probs_spread, method="gaussian", fwhm_myr=25
    )

    _, probs_unc_spread_smooth = smooth_curve(
        ages, probs_unc_spread, method="gaussian", fwhm_myr=25
    )

    # Feature mean + spread
    backarc_feat_mean_age = (
        backarc_data_filtered.groupby("age (Ma)")[feature]
        .mean()
        .reset_index()
        .sort_values("age (Ma)")
    )

    backarc_feat_spread_age = (
        backarc_data_filtered.groupby("age (Ma)")[feature]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name=feature)
        .sort_values("age (Ma)")
    )

    feats = backarc_feat_mean_age[feature].values.reshape(-1, 1)
    feats_spread = backarc_feat_spread_age[feature].values.reshape(-1, 1)

    _, feats_smooth = smooth_curve(
        ages, feats, method="gaussian", fwhm_myr=25
    )

    _, feats_spread_smooth = smooth_curve(
        ages, feats_spread, method="gaussian", fwhm_myr=25
    )

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    # Histogram
    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        color="gray",
        alpha=0.5,
        edgecolor="black",
        zorder=1,
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Probability axis
    ax2 = ax1.twinx()

    ax2.plot(
        ages_smooth,
        probs_smooth,
        color="orangered",
        linewidth=2,
        label="Mineralisation probability",
        zorder=2,
    )

    ax2.plot(
        ages_smooth,
        probs_unc_smooth,
        color="forestgreen",
        linewidth=2,
        label="Adjusted probability",
        zorder=2,
    )

    ax2.fill_between(
        ages_smooth.flatten(),
        (probs_smooth - probs_spread_smooth).flatten(),
        (probs_smooth + probs_spread_smooth).flatten(),
        color="orangered",
        alpha=0.2,
    )

    ax2.fill_between(
        ages_smooth.flatten(),
        (probs_unc_smooth - probs_unc_spread_smooth).flatten(),
        (probs_unc_smooth + probs_unc_spread_smooth).flatten(),
        color="forestgreen",
        alpha=0.2,
    )

    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel("Mineralisation probability", fontsize=14)

    # Feature axis (3rd axis)
    feature_label = format_feature_name(feature)
    uncertainty_label = "±1σ" if spread_method == "std" else "±MAD"
    feature_label_with_uncertainty = f"{feature_label} ({uncertainty_label})"

    ax3 = ax1.twinx()
    ax3.spines["right"].set_position(("outward", 60))

    ax3.plot(
        ages_smooth,
        feats_smooth,
        color="royalblue",
        linewidth=2,
        label=feature_label_with_uncertainty,
        zorder=2,
    )

    ax3.fill_between(
        ages_smooth.flatten(),
        (feats_smooth - feats_spread_smooth).flatten(),
        (feats_smooth + feats_spread_smooth).flatten(),
        color="royalblue",
        alpha=0.2,
    )

    ax3.set_ylabel(feature_label, fontsize=14)

    ax1.tick_params(labelsize=12)
    ax2.tick_params(labelsize=12)
    ax3.tick_params(labelsize=12)

    # Legend
    spread_label = "±1σ" if spread_method == "std" else "±MAD"

    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()

    labels2 = [f"{l} ({spread_label})" for l in labels2]

    lines = lines2 + lines3
    labels = labels2 + labels3

    ax3.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()
    
    clean_feature = clean_feature_name(feature)
    fig_path = os.path.join(figs_dir, f"backarc_prob_avg_trend_{clean_feature}.png")
    # if not os.path.exists(fig_path):
    #     plt.savefig(
    #         fig_path,
    #         dpi=300,
    #         bbox_inches="tight",
    #     )
    
    plt.show()

This cell is similar to the previous one, but instead of averaging all probability values, it averages only those greater than 0.5.

In [ ]:
# Get ages with probability > 0.5
backarc_prob_high = backarc_prob[backarc_prob["probability"] > 0.5]
backarc_prob_unc_high = backarc_prob[backarc_prob["probability_uncertainty"] > 0.5]

backarc_prob_high_mean_age = (
    backarc_prob_high.groupby("age (Ma)")["probability"]
    .mean()
    .reset_index()
)

backarc_prob_unc_high_mean_age = (
    backarc_prob_unc_high.groupby("age (Ma)")["probability_uncertainty"]
    .mean()
    .reset_index()
)

# Get all unique ages
all_ages = backarc_prob["age (Ma)"].unique()
complete_ages = pd.DataFrame({"age (Ma)": all_ages})

# Fill missing
backarc_prob_high_mean_age = complete_ages.merge(
    backarc_prob_high_mean_age,
    on="age (Ma)",
    how="left"
).fillna(backarc_prob_high_mean_age["probability"].mean())

backarc_prob_unc_high_mean_age = complete_ages.merge(
    backarc_prob_unc_high_mean_age,
    on="age (Ma)",
    how="left"
).fillna(backarc_prob_unc_high_mean_age["probability_uncertainty"].mean())

backarc_prob_high_mean_age = backarc_prob_high_mean_age.sort_values("age (Ma)")
backarc_prob_unc_high_mean_age = backarc_prob_unc_high_mean_age.sort_values("age (Ma)")

ages_high = backarc_prob_high_mean_age["age (Ma)"].values.reshape(-1, 1)
probs_high = backarc_prob_high_mean_age["probability"].values.reshape(-1, 1)
probs_unc_high = backarc_prob_unc_high_mean_age["probability_uncertainty"].values.reshape(-1, 1)

# Smooth mean once
ages_high_smooth, probs_high_smooth = smooth_curve(
    ages_high, probs_high, method="gaussian", fwhm_myr=25
)

_, probs_unc_high_smooth = smooth_curve(
    ages_high, probs_unc_high, method="gaussian", fwhm_myr=25
)

# Interactive plot
@interact
def show_map(
    feature=selected_features,
    spread_method=["std", "mad"],
    probability_type=[
        "Mineralisation probability",
        "Adjusted probability"
    ]
):
    if probability_type == "Mineralisation probability":
        prob_df = backarc_prob_high
        prob_col = "probability"
        prob_mean_smooth = probs_high_smooth
        prob_color = "orangered"
        prob_label = "Mineralisation probability"
    else:
        prob_df = backarc_prob_unc_high
        prob_col = "probability_uncertainty"
        prob_mean_smooth = probs_unc_high_smooth
        prob_color = "forestgreen"
        prob_label = "Adjusted probability"

    # Probability spread
    prob_spread_age = (
        prob_df.groupby("age (Ma)")[prob_col]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name=prob_col)
    )

    prob_spread_age = complete_ages.merge(
        prob_spread_age,
        on="age (Ma)",
        how="left"
    ).fillna(prob_spread_age[prob_col].mean())

    prob_spread_age = prob_spread_age.sort_values("age (Ma)")

    prob_spread = prob_spread_age[prob_col].values.reshape(-1, 1)

    _, prob_spread_smooth = smooth_curve(
        ages_high, prob_spread, method="gaussian", fwhm_myr=25
    )

    # Feature mean + spread for selected probability type
    backarc_feat_high = prob_df.merge(
        backarc_data_filtered,
        left_index=True,
        right_index=True,
        suffixes=('', '_drop')
    ).filter(regex='^(?!.*_drop)')

    # Feature mean
    backarc_feat_high_mean_age = (
        backarc_feat_high.groupby("age (Ma)")[feature]
        .mean()
        .reset_index()
    )

    backarc_feat_high_mean_age = complete_ages.merge(
        backarc_feat_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(backarc_feat_high_mean_age[feature].mean())

    # Feature spread
    backarc_feat_high_spread_age = (
        backarc_feat_high.groupby("age (Ma)")[feature]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name=feature)
    )

    backarc_feat_high_spread_age = complete_ages.merge(
        backarc_feat_high_spread_age,
        on="age (Ma)",
        how="left"
    ).fillna(backarc_feat_high_spread_age[feature].mean())

    # Sort
    backarc_feat_high_mean_age = backarc_feat_high_mean_age.sort_values("age (Ma)")
    backarc_feat_high_spread_age = backarc_feat_high_spread_age.sort_values("age (Ma)")

    # Arrays
    feat_vals = backarc_feat_high_mean_age[feature].values.reshape(-1, 1)
    feat_spread = backarc_feat_high_spread_age[feature].values.reshape(-1, 1)

    # Smooth
    _, feat_vals_smooth = smooth_curve(
        ages_high, feat_vals, method="gaussian", fwhm_myr=25
    )

    _, feat_spread_smooth = smooth_curve(
        ages_high, feat_spread, method="gaussian", fwhm_myr=25
    )

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        color="gray",
        alpha=0.5,
        edgecolor="black"
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Probability axis
    ax2 = ax1.twinx()

    ax2.plot(
        ages_high_smooth,
        prob_mean_smooth,
        color=prob_color,
        linewidth=2,
        label=prob_label,
    )

    ax2.fill_between(
        ages_high_smooth.flatten(),
        (prob_mean_smooth - prob_spread_smooth).flatten(),
        (prob_mean_smooth + prob_spread_smooth).flatten(),
        color=prob_color,
        alpha=0.2,
    )

    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0.5, 1)
    ax2.set_ylabel(f"{prob_label} (P > 0.5)", fontsize=14)

    # Feature axis
    feature_label = format_feature_name(feature)
    uncertainty_label = "±1σ" if spread_method == "std" else "±MAD"
    feature_label_with_uncertainty = f"{feature_label} ({uncertainty_label})"

    ax3 = ax1.twinx()
    ax3.spines["right"].set_position(("outward", 60))

    ax3.plot(
        ages_high_smooth,
        feat_vals_smooth,
        color="royalblue",
        linewidth=2,
        label=feature_label_with_uncertainty
    )

    ax3.fill_between(
        ages_high_smooth.flatten(),
        (feat_vals_smooth - feat_spread_smooth).flatten(),
        (feat_vals_smooth + feat_spread_smooth).flatten(),
        color="royalblue",
        alpha=0.2,
    )

    ax3.set_ylabel(feature_label, fontsize=14)

    ax1.tick_params(labelsize=12)
    ax2.tick_params(labelsize=12)
    ax3.tick_params(labelsize=12)

    # Legend
    spread_label = "±1σ" if spread_method == "std" else "±MAD"

    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()

    labels2 = [f"{l} ({spread_label})" for l in labels2]

    lines = lines2 + lines3
    labels = labels2 + labels3

    ax3.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()

    clean_feature = clean_feature_name(feature)
    clean_prob_type = clean_feature_name(probability_type)
    fig_path = os.path.join(
        figs_dir,
        f"backarc_prob_avg_trend_high_{clean_prob_type}_{clean_feature}.png"
    )

    # if not os.path.exists(fig_path):
    #     plt.savefig(
    #         fig_path,
    #         dpi=300,
    #         bbox_inches="tight",
    #     )

    plt.show()

This cell allows the user to plot the average values of probability, uncertainty, and adjusted probability by uncertainty.

In [ ]:
@interact
def show_uncertainty(spread_method=["std", "mad", "none"]):
    # Mean probability and uncertainty
    backarc_prob_mean_age = (
        backarc_prob.groupby("age (Ma)")["probability"]
        .mean()
        .reset_index()
    )

    backarc_unc_mean_age = (
        backarc_prob.groupby("age (Ma)")["uncertainty"]
        .mean()
        .reset_index()
    )

    backarc_prob_unc_mean_age = (
        backarc_prob.groupby("age (Ma)")["probability_uncertainty"]
        .mean()
        .reset_index()
    )

    backarc_prob_mean_age = complete_ages.merge(
        backarc_prob_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(backarc_prob_mean_age["probability"].mean())

    backarc_unc_mean_age = complete_ages.merge(
        backarc_unc_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(backarc_unc_mean_age["uncertainty"].mean())

    backarc_prob_unc_mean_age = complete_ages.merge(
        backarc_prob_unc_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(backarc_prob_unc_mean_age["probability_uncertainty"].mean())

    backarc_prob_mean_age = backarc_prob_mean_age.sort_values("age (Ma)")
    backarc_unc_mean_age = backarc_unc_mean_age.sort_values("age (Ma)")
    backarc_prob_unc_mean_age = backarc_prob_unc_mean_age.sort_values("age (Ma)")

    probs = backarc_prob_mean_age["probability"].values.reshape(-1, 1)
    unc = backarc_unc_mean_age["uncertainty"].values.reshape(-1, 1)
    probs_unc = backarc_prob_unc_mean_age["probability_uncertainty"].values.reshape(-1, 1)

    # Spread (only if needed)
    if spread_method != "none":

        prob_spread_age = (
            backarc_prob.groupby("age (Ma)")["probability"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="probability")
        )

        unc_spread_age = (
            backarc_prob.groupby("age (Ma)")["uncertainty"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="uncertainty")
        )

        prob_unc_spread_age = (
            backarc_prob.groupby("age (Ma)")["probability_uncertainty"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="probability_uncertainty")
        )

        # Fill missing
        prob_spread_age = complete_ages.merge(prob_spread_age, on="age (Ma)", how="left")\
            .fillna(prob_spread_age["probability"].mean())

        unc_spread_age = complete_ages.merge(unc_spread_age, on="age (Ma)", how="left")\
            .fillna(unc_spread_age["uncertainty"].mean())

        prob_unc_spread_age = complete_ages.merge(prob_unc_spread_age, on="age (Ma)", how="left")\
            .fillna(prob_unc_spread_age["probability_uncertainty"].mean())

    ages_unc = complete_ages.sort_values("age (Ma)")["age (Ma)"].values.reshape(-1, 1)

    # Smooth means
    ages_unc_smooth, probs_smooth = smooth_curve(
        ages_unc, probs, method="gaussian", fwhm_myr=25
    )

    _, unc_smooth = smooth_curve(
        ages_unc, unc, method="gaussian", fwhm_myr=25
    )

    _, probs_unc_smooth = smooth_curve(
        ages_unc, probs_unc, method="gaussian", fwhm_myr=25
    )

    # Smooth spreads (if enabled)
    if spread_method != "none":
        probs_spread = prob_spread_age.sort_values("age (Ma)")["probability"].values.reshape(-1, 1)
        unc_spread = unc_spread_age.sort_values("age (Ma)")["uncertainty"].values.reshape(-1, 1)
        probs_unc_spread = prob_unc_spread_age.sort_values("age (Ma)")["probability_uncertainty"].values.reshape(-1, 1)

        _, probs_spread_smooth = smooth_curve(ages_unc, probs_spread, method="gaussian", fwhm_myr=25)
        _, unc_spread_smooth = smooth_curve(ages_unc, unc_spread, method="gaussian", fwhm_myr=25)
        _, probs_unc_spread_smooth = smooth_curve(ages_unc, probs_unc_spread, method="gaussian", fwhm_myr=25)

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        color="gray",
        alpha=0.5,
        edgecolor="black"
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Label text
    uncertainty_label = "" if spread_method == "none" else ("±1σ" if spread_method == "std" else "±MAD")

    # Probability axis
    ax2 = ax1.twinx()
    ax2.plot(ages_unc_smooth, probs_smooth,
             color="orangered", linewidth=2,
             label=f"Mineralisation probability {f'({uncertainty_label})' if uncertainty_label else ''}")

    ax2.plot(ages_unc_smooth, unc_smooth,
             color="royalblue", linewidth=2,
             label=f"Uncertainty {f'({uncertainty_label})' if uncertainty_label else ''}")

    ax2.plot(ages_unc_smooth, probs_unc_smooth,
             color="forestgreen", linewidth=2,
             label=f"Adjusted probability {f'({uncertainty_label})' if uncertainty_label else ''}")

    if spread_method != "none":
        ax2.fill_between(
            ages_unc_smooth.flatten(),
            (probs_smooth - probs_spread_smooth).flatten(),
            (probs_smooth + probs_spread_smooth).flatten(),
            color="orangered",
            alpha=0.2
        )

        ax2.fill_between(
            ages_unc_smooth.flatten(),
            (unc_smooth - unc_spread_smooth).flatten(),
            (unc_smooth + unc_spread_smooth).flatten(),
            color="royalblue",
            alpha=0.2
        )

        ax2.fill_between(
            ages_unc_smooth.flatten(),
            (probs_unc_smooth - probs_unc_spread_smooth).flatten(),
            (probs_unc_smooth + probs_unc_spread_smooth).flatten(),
            color="forestgreen",
            alpha=0.2
        )
    
    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel("Mineralisation probability / uncertainty", fontsize=14)

    ax1.tick_params(labelsize=12)
    ax2.tick_params(labelsize=12)

    # Legend
    lines, labels = [], []    
    l, lab = ax2.get_legend_handles_labels()
    lines += l
    labels += lab

    ax2.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()
    
    fig_path = os.path.join(figs_dir, "backarc_prob_unc_avg_trend.png")
    # if not os.path.exists(fig_path):
    #     fig.savefig(
    #         fig_path,
    #         dpi=300,
    #         bbox_inches="tight",
    #     )
    
    plt.show()